# Load and query the cleaned Telco CSV with MySQL

Install the project dependencies with `pip install -r requirements.txt`. This workflow uses SQLAlchemy with the PyMySQL driver. Create the target MySQL database before running the notebook (for example, `CREATE DATABASE apexplanet;`).

Copy the project-root `.env.example` to `.env` and set `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_PASSWORD`, and `DB_NAME` there. `db_utils.py` loads `.env` when creating the engine; existing process environment variables take precedence. Never put credentials in this notebook or commit `.env`. The CSV loader defaults to `if_exists="fail"` and will not overwrite an existing table.

In [15]:
import sys
from pathlib import Path

from sqlalchemy import text

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "db_utils.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find db_utils.py in this directory or its parents.")
sys.path.insert(0, str(project_root))

from db_utils import create_mysql_engine, load_csv_to_mysql, read_sql_dataframe

csv_path = project_root / "data" / "processed" / "telco_churn_clean.csv"
engine = create_mysql_engine()

## Verify the MySQL connection

In [16]:
with engine.connect() as connection:
    mysql_version = connection.execute(text("SELECT VERSION()")).scalar_one()
print(f"Connected to MySQL {mysql_version}")

Connected to MySQL 8.0.46


## SQL fundamentals practice

The sample query above demonstrates `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, and bound parameters. The practice cell below adds joins, aggregates, subqueries, CTEs, and window functions. CTEs and window functions require MySQL 8.0 or later.

## Advanced SQL business questions

This Telco dataset is a single customer snapshot: it has no order dates or product sales. Monthly sales trends and product-category sales therefore cannot be calculated. The exercises use `monthly_charges` as a recurring-revenue snapshot, `total_charges` as lifetime billed charges, internet service as a service category, and churn status for retention.

In [18]:
query = """
SELECT customer_id, tenure, contract, monthly_charges, churn
FROM telco_churn
WHERE tenure >= :minimum_tenure
  AND churn = :churn_status
ORDER BY monthly_charges DESC
LIMIT :row_limit
"""
results = read_sql_dataframe(
    engine,
    query,
    params={"minimum_tenure": 12, "churn_status": "Yes", "row_limit": 10},
)
results

,customer_id,tenure,contract,monthly_charges,churn
0,8199-ZLLSA,67,One year,118.35,Yes
1,2889-FPWRM,72,One year,117.80,Yes
2,2302-ANTDP,48,Month-to-month,117.45,Yes
3,9053-JZFKV,67,Two year,116.20,Yes
4,1444-VVSGW,70,One year,115.65,Yes
5,0201-OAMXR,70,One year,115.55,Yes
6,4361-BKAXE,41,Month-to-month,114.50,Yes
7,1555-DJEQW,70,Two year,114.20,Yes
8,9158-VCTQB,41,Month-to-month,113.60,Yes
9,7279-BUYWN,41,Month-to-month,113.20,Yes


In [21]:
from IPython.display import Markdown, display

sql_exercises = {
    "GROUP BY, HAVING, and aggregates": """
        SELECT contract,
               COUNT(*) AS customer_count,
               SUM(total_charges) AS lifetime_charges,
               AVG(monthly_charges) AS avg_monthly_charges,
               MIN(monthly_charges) AS min_monthly_charges,
               MAX(monthly_charges) AS max_monthly_charges,
               AVG(churn = 'Yes') AS churn_rate
        FROM telco_churn
        GROUP BY contract
        HAVING COUNT(*) >= 100
        ORDER BY lifetime_charges DESC
    """,
    "INNER JOIN: customers who churned": """
        WITH churned_customers AS (
            SELECT customer_id, churn
            FROM telco_churn
            WHERE churn = 'Yes'
        )
        SELECT t.customer_id, t.contract, c.churn
        FROM telco_churn AS t
        INNER JOIN churned_customers AS c ON c.customer_id = t.customer_id
        ORDER BY t.customer_id
        LIMIT 20
    """,
    "LEFT JOIN: all customers, with churn-cohort matches": """
        WITH churned_customers AS (
            SELECT customer_id, churn
            FROM telco_churn
            WHERE churn = 'Yes'
        )
        SELECT t.customer_id, t.contract, c.churn AS churn_cohort_match
        FROM telco_churn AS t
        LEFT JOIN churned_customers AS c ON c.customer_id = t.customer_id
        ORDER BY t.customer_id
        LIMIT 20
    """,
    "RIGHT JOIN: same result with the preserved table on the right": """
        WITH churned_customers AS (
            SELECT customer_id, churn
            FROM telco_churn
            WHERE churn = 'Yes'
        )
        SELECT t.customer_id, t.contract, c.churn AS churn_cohort_match
        FROM churned_customers AS c
        RIGHT JOIN telco_churn AS t ON t.customer_id = c.customer_id
        ORDER BY t.customer_id
        LIMIT 20
    """,
    "FULL OUTER JOIN emulation for MySQL using UNION ALL": """
        WITH churned_customers AS (
            SELECT customer_id, churn
            FROM telco_churn
            WHERE churn = 'Yes'
        )
        SELECT t.customer_id, t.contract, c.churn AS churn_cohort_match
        FROM telco_churn AS t
        LEFT JOIN churned_customers AS c ON c.customer_id = t.customer_id
        UNION ALL
        SELECT c.customer_id, NULL AS contract, c.churn AS churn_cohort_match
        FROM churned_customers AS c
        LEFT JOIN telco_churn AS t ON t.customer_id = c.customer_id
        WHERE t.customer_id IS NULL
        LIMIT 20
    """,
    "Subquery: customers billed above the overall average": """
        SELECT customer_id, contract, monthly_charges
        FROM telco_churn
        WHERE monthly_charges > (
            SELECT AVG(monthly_charges) FROM telco_churn
        )
        ORDER BY monthly_charges DESC
        LIMIT 10
    """,
    "CTE: retention rate by contract": """
        WITH contract_retention AS (
            SELECT contract,
                   COUNT(*) AS customer_count,
                   SUM(churn = 'No') AS retained_customers,
                   100.0 * SUM(churn = 'No') / COUNT(*) AS retention_rate_pct
            FROM telco_churn
            GROUP BY contract
        )
        SELECT *
        FROM contract_retention
        ORDER BY retention_rate_pct DESC
    """,
    "Window functions: ROW_NUMBER, RANK, LAG, and LEAD": """
        SELECT customer_id, contract, tenure, total_charges,
               ROW_NUMBER() OVER (
                   PARTITION BY contract ORDER BY total_charges DESC
               ) AS revenue_row_number,
               RANK() OVER (
                   PARTITION BY contract ORDER BY total_charges DESC
               ) AS revenue_rank,
               LAG(total_charges) OVER (
                   PARTITION BY contract ORDER BY tenure, customer_id
               ) AS previous_customer_charges,
               LEAD(total_charges) OVER (
                   PARTITION BY contract ORDER BY tenure, customer_id
               ) AS next_customer_charges
        FROM telco_churn
        ORDER BY contract, revenue_row_number
        LIMIT 20
    """,
    "Top 10 customers by lifetime billed charges (revenue proxy)": """
        SELECT customer_id, total_charges, tenure, contract
        FROM telco_churn
        ORDER BY total_charges DESC
        LIMIT 10
    """,
    "Service-category performance (revenue proxy and retention)": """
        SELECT internet_service AS service_category,
               COUNT(*) AS customer_count,
               SUM(total_charges) AS lifetime_charges,
               AVG(monthly_charges) AS avg_monthly_charges,
               100.0 * SUM(churn = 'No') / COUNT(*) AS retention_rate_pct
        FROM telco_churn
        GROUP BY internet_service
        ORDER BY lifetime_charges DESC
    """,
    "Current recurring-charge snapshot by contract (not a monthly trend)": """
        SELECT contract,
               SUM(monthly_charges) AS current_recurring_charges
        FROM telco_churn
        GROUP BY contract
        ORDER BY current_recurring_charges DESC
    """,
    "Five-tenure rolling average and cumulative customer count": """
        WITH tenure_summary AS (
            SELECT tenure,
                   COUNT(*) AS customer_count,
                   AVG(monthly_charges) AS avg_monthly_charges
            FROM telco_churn
            GROUP BY tenure
        )
        SELECT tenure, customer_count, avg_monthly_charges,
               AVG(avg_monthly_charges) OVER (
                   ORDER BY tenure ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
               ) AS five_tenure_rolling_avg,
               SUM(customer_count) OVER (
                   ORDER BY tenure ROWS UNBOUNDED PRECEDING
               ) AS cumulative_customer_count
        FROM tenure_summary
        ORDER BY tenure
    """,
}

for exercise_name, exercise_query in sql_exercises.items():
    display(Markdown(f"### {exercise_name}"))
    display(read_sql_dataframe(engine, exercise_query))

### GROUP BY, HAVING, and aggregates

,contract,customer_count,lifetime_charges,avg_monthly_charges,min_monthly_charges,max_monthly_charges,churn_rate
0,Two year,1695,6297228.450,60.770413,18.40,118.75,0.0283
1,Month-to-month,3875,5305861.500,66.398490,18.75,117.45,0.4271
2,One year,1473,4468450.975,65.048608,18.25,118.60,0.1127


### INNER JOIN: customers who churned

,customer_id,contract,churn
0,0004-TLHLJ,Month-to-month,Yes
1,0011-IGKFF,Month-to-month,Yes
2,0013-EXCHZ,Month-to-month,Yes
3,0022-TCJCI,One year,Yes
4,0023-HGHWL,Month-to-month,Yes
5,0023-XUOPT,Month-to-month,Yes
6,0031-PVLZI,Month-to-month,Yes
7,0032-PGELS,Month-to-month,Yes
8,0067-DKWBL,Month-to-month,Yes
9,0093-XWZFY,Month-to-month,Yes


### LEFT JOIN: all customers, with churn-cohort matches

...

,service_category,customer_count,lifetime_charges,avg_monthly_charges,retention_rate_pct
0,Fiber optic,3096,9923622.950,91.500129,58.10724
1,DSL,2421,5128398.225,58.102169,81.04089
2,No,1526,1019519.750,21.079194,92.59502


### Current recurring-charge snapshot by contract (not a monthly trend)

,contract,current_recurring_charges
0,Month-to-month,257294.15
1,Two year,103005.85
2,One year,95816.60


### Five-tenure rolling average and cumulative customer count

,tenure,customer_count,avg_monthly_charges,five_tenure_rolling_avg,cumulative_customer_count
0,0,11,41.418182,41.418182,11.0
1,1,613,50.485808,45.951995,624.0
2,2,238,57.206303,49.703431,862.0
3,3,200,58.015000,51.781323,1062.0
4,4,176,57.432670,52.911592,1238.0
...,...,...,...,...,...
68,68,100,73.321000,75.589464,6297.0
69,69,95,70.823158,74.597970,6392.0
70,70,119,76.378992,73.784163,6511.0
71,71,170,73.735588,73.318584,6681.0


## Views and query optimization

Create a reusable contract summary view. `IF NOT EXISTS` makes the cell safe to re-run without replacing an existing view.

```sql
CREATE VIEW IF NOT EXISTS telco_contract_summary AS
SELECT contract, COUNT(*) AS customer_count,
       AVG(monthly_charges) AS avg_monthly_charges,
       100.0 * SUM(churn = 'No') / COUNT(*) AS retention_rate_pct
FROM telco_churn
GROUP BY contract;

SELECT * FROM telco_contract_summary;
EXPLAIN SELECT customer_id, monthly_charges
FROM telco_churn WHERE contract = 'One year' AND churn = 'No';
```

Use `EXPLAIN` before and after testing an index. For this filter, a candidate is `CREATE INDEX idx_telco_contract_churn ON telco_churn (contract, churn);` (run once only). Indexes add storage and write cost; keep an index only when the execution plan and workload justify it.

In [20]:
engine.dispose()